# Instantiation methods

In [1]:
# Import all of the packages we will use throughout this notebook
import numpy as np
import pandas as pd
import lkdata as ld

%load_ext autoreload
%autoreload 2

In [2]:
# Optional display options
np.set_printoptions(threshold=10)  # limiting to printing up to 10 values at a time
pd.set_option("display.max_columns", 15)  # limiting to printing up to 15 columns

In [3]:
ntime = 120

# Generate times and random corrections
times = np.linspace(0, 24, ntime)
time_corr = np.random.random(times.shape)

# Generate some random data
series_data = np.random.standard_normal(times.shape)

# 1 - Time
- via `time_indices` (recommended)
  - a dictionary of time arrays
- via `index`, a la `pandas`
  - arraylike
  - pandas index object
- None or via `ntime` - automatically generates a RangeIndex called "time_index". Specifying `ntime` should be unnecessary.

**Notes:**
- A ranged index called "time_index" is automatically created unless an index with the same name is given.
- Duplicate indices are dropped, defaulting to the first in order given.

## 1.1 - Recommended: `time_indices`

It is easy to give any number of named indices via dictionary with a `str: array-like` pair

In [4]:
series = ld.DataSeries(
    series_data, time_indices={"fake_jd": times, "corrected_time": times + time_corr}
)
series.describe_series()

📉 DataSeries (120,), Uncertainty: False (120,) (ntime)


Time indices available: ['time_index', 'fake_jd', 'corrected_time']
  time_index     :	[  0   1 ... 118 119]
  fake_jd        :	[ 0.          0.20168067 ... 23.79831933 24.        ]
  corrected_time :	[ 0.57803325  0.45627162 ... 24.19069131 24.51886407]


## 1.2 - Array-like via `index`

This is the most Pandas-like and is easy for giving a single array for an index. It is automatically named "given_index" and the ranged "time_index" is created.


In [5]:
series = ld.DataSeries(series_data, index=times)
series.describe_series()

📉 DataSeries (120,), Uncertainty: False (120,) (ntime)


Time indices available: ['time_index', 'given_index']
  time_index  :	[  0   1 ... 118 119]
  given_index :	[ 0.          0.20168067 ... 23.79831933 24.        ]


## 1.3 - Pandas `Index` via `index`
This is the easiest when working with an existing pandas object with an index property (which includes the lkdata objects). Index names carry into the new index. "time_index" is created unless it exists in the given index.

### 1.3.1 - Using an index from an existing object

In [7]:
df = pd.DataFrame(series_data, index=times)
df.index.name = "time_index"
series = ld.DataSeries(series_data, index=df.index)
series.describe_series()

📉 DataSeries (120,), Uncertainty: False (120,) (ntime)


Time indices available: ['time_index']
  time_index  :	[ 0.          0.20168067 ... 23.79831933 24.        ]


### 1.3.2 - Creating an index beforehand
It's also possible to generate pandas indices directly, see the [pandas Index objects documentation](https://pandas.pydata.org/docs/reference/indexing.html) for all options.
This enables very fine-grained control over the indices of lkdata objects.

In [10]:
index = pd.TimedeltaIndex(pd.to_timedelta(times, unit="d"), name="time")
series = ld.DataSeries(series_data, index=index)
# time_indices can also accept pd.Index objects, but will be renamed to the given key
# series = lk.DataSeries(series_data, time_indices={"time_diffname": index})
series.describe_series()

📉 DataSeries (120,), Uncertainty: False (120,) (ntime)


Time indices available: ['time_index', 'time']
  time_index  :	[  0   1 ... 118 119]
  time        :	[               0   17425210084032 ... 2056174789915967 2073600000000000]


For multiple indices, see [pandas.MultiIndex](https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.html).

In [ ]:
index = pd.MultiIndex.from_arrays(
    [times, times + time_corr], names=["custom", "custom_corrected"]
)

# Another example, combining multiple pandas Index objects. Note that names carry through:
# rindex = pd.RangeIndex(ntime, name="time_index")
# dtindex = pd.TimedeltaIndex(pd.to_timedelta(times, unit='d'), name="time")
# index = pd.MultiIndex.from_arrays([rindex, dtindex])

series = ld.DataSeries(
    series_data, index=index
)  # time_indices cannot accept a MultiIndex

series.describe_series()

📉 DataSeries (120,), Uncertainty: False (120,) (ntime)

Uncertainty:
	uncertainty	:	Uncertainty(None)

Time indices available: ['time_index', 'custom', 'custom_corrected']
	time_index       :	[  0   1 ... 118 119]
	custom           :	[ 0.          0.20168067 ... 23.79831933 24.        ]
	custom_corrected :	[ 0.16068803  1.0507715  ... 23.84324525 24.46017741]


# 2 - Space
- [2.1](#21---recommended-give-dictionaries-to-row_indices-and-col_indices) - `row_indices` and `col_indices` (recommended)
  - Values can be a list of unique ordered indices, with lengths `nrow` and `ncol` respectively (`cube` only), or
  - a full list of coordinate values
    - each with length `nrow`$\times$`ncol` for `cube` objects (defined explicitly or based on the shape of the data)
    - any length for `SeriesCollection`, lengths must match and be equal to the number of series present
- `nrow` and `ncol` - automatically generates a `RangeIndex` for each. (`Cube` only with flattened data)
- `columns` (`Cube` and `SeriesCollection`)
  -  For `Cube`, this will only work properly if "row" and "col" are in the column names to properly order data in the array representation.
  -  For `SeriesCollection` no interpretation of the columns will be attempted.
- None
  - for `Cube` generates `RangeIndex` if data is given as an (ntime, nrow, ncol) `ArrayLike`, otherwise raises an error where the shape cannot be interpreted as a cube.
  - generates a `RangeIndex` for the number or series present.

In [31]:
ntime = 120
nrow = 10
ncol = 12

# Generate times and random corrections
times = np.linspace(0, 24, ntime)

# Generate some random data
data = np.random.standard_normal((ntime, nrow, ncol))

## 2.1 - Recommended: give dictionaries to `row_indices` and `col_indices`
Give indices for rows and columns via `row_indices` and `col_indices`.
Multiple row and column levels can be defined with different coordinate systems.
There are no naming requirements (save for a few reserved names) when provided this way as rows and column information is provided and stored separately.

### 2.1.1 - Arrays given as ranges of unique values
Useful where a coordinate system or a pixel reference point is given, rather than coordinates.

In [32]:
# Arrays given as ranges of unique values
# Useful where only a reference pixel position is given
row_arr = np.arange(100, 100 + nrow)
col_arr = np.arange(200, 200 + ncol)

row_pos = row_arr * 1.8 - 13.5  # fake coordinate transformation
col_pos = col_arr * 0.8 - 1.35  # fake coordinate transformation
row_arr, col_arr

(array([100, 101, 102, 103, 104, 105, 106, 107, 108, 109]),
 array([200, 201, 202, ..., 209, 210, 211]))

In [33]:
cube = ld.DataCube(
    data,
    time_indices={"time": times},
    row_indices={"pix_row": row_arr, "row_pos": row_pos},
    col_indices={"pix_col": col_arr, "col_pos": col_pos},
)
pd.DataFrame(cube)

,series,0,1,2,3,4,5,6,...,113,114,115,116,117,118,119
,pix_row,100,100,100,100,100,100,100,...,109,109,109,109,109,109,109
,row_pos,166.5,166.5,166.5,166.5,166.5,166.5,166.5,...,182.7,182.7,182.7,182.7,182.7,182.7,182.7
,pix_col,200,201,202,203,204,205,206,...,205,206,207,208,209,210,211
,col_pos,158.65,159.45,160.25,161.05,161.85,162.65,163.45,...,162.65,163.45,164.25,165.05,165.85,166.65,167.45
time_index,time,,,,,,,,,,,,,,,
0,0.000000,0.451025,-0.012730,0.118395,-0.123785,0.368806,0.812296,-1.491025,...,0.542614,0.538944,1.126243,-0.693383,0.168361,-1.926612,-0.747363
1,0.201681,-0.809616,0.694303,1.738103,-0.846800,-0.106610,-1.089616,-0.010108,...,0.468733,0.960319,1.080917,-1.763136,0.581766,-0.629591,-1.067231
2,0.403361,-2.159283,-0.347606,-1.886857,1.021704,0.159404,0.452730,-0.070942,...,0.285698,0.575760,-1.265924,-0.829648,0.758606,0.696689,0.255458
3,0.605042,0.808381,1.468899,-1.657832,1.457362,-0.572793,1.401293,-0.894599,...,1.115110,0.585800,0.804090,1.805249,-2.014900,0.644703,0.118226
4,0.806723,-0.116969,0.485642,-1.661645,-0.095260,0.652312,-1.422831,0.398717,...,2.293649,0.156011,-0.579476,1.150948,-0.761896,-1.584643,0.281398


### 2.1.2 - Arrays given as coordinates

In [34]:
row_arr = np.repeat(np.arange(100, 100 + nrow), ncol)
col_arr = np.tile(np.arange(200, 200 + ncol), nrow)

row_pos = row_arr * 1.8 - 13.5  # fake coordinate transformation
col_pos = col_arr * 0.8 - 1.35  # fake coordinate transformation
np.array(list(zip(row_arr, col_arr)))

array([[100, 200],
       [100, 201],
       [100, 202],
       ...,
       [109, 209],
       [109, 210],
       [109, 211]])

In [35]:
cube = ld.DataCube(
    data,
    time_indices={"time": times},
    row_indices={"pix_row": row_arr, "row_pos": row_pos},
    col_indices={"pix_col": col_arr, "col_pos": col_pos},
)
pd.DataFrame(cube)

,series,0,1,2,3,4,5,6,...,113,114,115,116,117,118,119
,pix_row,100,100,100,100,100,100,100,...,109,109,109,109,109,109,109
,row_pos,166.5,166.5,166.5,166.5,166.5,166.5,166.5,...,182.7,182.7,182.7,182.7,182.7,182.7,182.7
,pix_col,200,201,202,203,204,205,206,...,205,206,207,208,209,210,211
,col_pos,158.65,159.45,160.25,161.05,161.85,162.65,163.45,...,162.65,163.45,164.25,165.05,165.85,166.65,167.45
time_index,time,,,,,,,,,,,,,,,
0,0.000000,0.451025,-0.012730,0.118395,-0.123785,0.368806,0.812296,-1.491025,...,0.542614,0.538944,1.126243,-0.693383,0.168361,-1.926612,-0.747363
1,0.201681,-0.809616,0.694303,1.738103,-0.846800,-0.106610,-1.089616,-0.010108,...,0.468733,0.960319,1.080917,-1.763136,0.581766,-0.629591,-1.067231
2,0.403361,-2.159283,-0.347606,-1.886857,1.021704,0.159404,0.452730,-0.070942,...,0.285698,0.575760,-1.265924,-0.829648,0.758606,0.696689,0.255458
3,0.605042,0.808381,1.468899,-1.657832,1.457362,-0.572793,1.401293,-0.894599,...,1.115110,0.585800,0.804090,1.805249,-2.014900,0.644703,0.118226
4,0.806723,-0.116969,0.485642,-1.661645,-0.095260,0.652312,-1.422831,0.398717,...,2.293649,0.156011,-0.579476,1.150948,-0.761896,-1.584643,0.281398


In [36]:
series_collection = ld.DataSeriesCollection(
    data.reshape(120, 120),
    row_indices={"row": row_arr},
    col_indices={"col": col_arr},
)
series_collection

series,0,1,2,3,4,5,6,...,113,114,115,116,117,118,119
row,100,100,100,100,100,100,100,...,109,109,109,109,109,109,109
col,200,201,202,203,204,205,206,...,205,206,207,208,209,210,211
time_index,,,,,,,,,,,,,,,
0,0.451025,-0.012730,0.118395,-0.123785,0.368806,0.812296,-1.491025,...,0.542614,0.538944,1.126243,-0.693383,0.168361,-1.926612,-0.747363
1,-0.809616,0.694303,1.738103,-0.846800,-0.106610,-1.089616,-0.010108,...,0.468733,0.960319,1.080917,-1.763136,0.581766,-0.629591,-1.067231
2,-2.159283,-0.347606,-1.886857,1.021704,0.159404,0.452730,-0.070942,...,0.285698,0.575760,-1.265924,-0.829648,0.758606,0.696689,0.255458
3,0.808381,1.468899,-1.657832,1.457362,-0.572793,1.401293,-0.894599,...,1.115110,0.585800,0.804090,1.805249,-2.014900,0.644703,0.118226
4,-0.116969,0.485642,-1.661645,-0.095260,0.652312,-1.422831,0.398717,...,2.293649,0.156011,-0.579476,1.150948,-0.761896,-1.584643,0.281398
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,2.111950,1.555535,0.472429,0.235726,0.270339,0.411209,-1.186713,...,0.498939,0.422257,1.173478,1.003721,-0.506200,0.872855,-0.669145


## 2.2 - If coordinates aren't particularly important
If the data is given as an array with shape (ntime, nrow, ncol) and coordinates are not particularly important,
row and column information can be omitted and indices will be generated for both along with a `RangeIndex` to keep
track of each series.

In [37]:
cube = ld.DataCube(
    data,
    time_indices={"time": times},
)
pd.DataFrame(cube)

,series,0,1,2,3,4,5,6,...,113,114,115,116,117,118,119
,row,0,0,0,0,0,0,0,...,9,9,9,9,9,9,9
,col,0,1,2,3,4,5,6,...,5,6,7,8,9,10,11
time_index,time,,,,,,,,,,,,,,,
0,0.000000,0.451025,-0.012730,0.118395,-0.123785,0.368806,0.812296,-1.491025,...,0.542614,0.538944,1.126243,-0.693383,0.168361,-1.926612,-0.747363
1,0.201681,-0.809616,0.694303,1.738103,-0.846800,-0.106610,-1.089616,-0.010108,...,0.468733,0.960319,1.080917,-1.763136,0.581766,-0.629591,-1.067231
2,0.403361,-2.159283,-0.347606,-1.886857,1.021704,0.159404,0.452730,-0.070942,...,0.285698,0.575760,-1.265924,-0.829648,0.758606,0.696689,0.255458
3,0.605042,0.808381,1.468899,-1.657832,1.457362,-0.572793,1.401293,-0.894599,...,1.115110,0.585800,0.804090,1.805249,-2.014900,0.644703,0.118226
4,0.806723,-0.116969,0.485642,-1.661645,-0.095260,0.652312,-1.422831,0.398717,...,2.293649,0.156011,-0.579476,1.150948,-0.761896,-1.584643,0.281398
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,23.193277,2.111950,1.555535,0.472429,0.235726,0.270339,0.411209,-1.186713,...,0.498939,0.422257,1.173478,1.003721,-0.506200,0.872855,-0.669145


In [40]:
series_collection = ld.DataSeriesCollection(
    data.reshape(120, 120),
)
series_collection

series,0,1,2,3,4,5,6,...,113,114,115,116,117,118,119
time_index,,,,,,,,,,,,,,,
0,0.451025,-0.012730,0.118395,-0.123785,0.368806,0.812296,-1.491025,...,0.542614,0.538944,1.126243,-0.693383,0.168361,-1.926612,-0.747363
1,-0.809616,0.694303,1.738103,-0.846800,-0.106610,-1.089616,-0.010108,...,0.468733,0.960319,1.080917,-1.763136,0.581766,-0.629591,-1.067231
2,-2.159283,-0.347606,-1.886857,1.021704,0.159404,0.452730,-0.070942,...,0.285698,0.575760,-1.265924,-0.829648,0.758606,0.696689,0.255458
3,0.808381,1.468899,-1.657832,1.457362,-0.572793,1.401293,-0.894599,...,1.115110,0.585800,0.804090,1.805249,-2.014900,0.644703,0.118226
4,-0.116969,0.485642,-1.661645,-0.095260,0.652312,-1.422831,0.398717,...,2.293649,0.156011,-0.579476,1.150948,-0.761896,-1.584643,0.281398
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,2.111950,1.555535,0.472429,0.235726,0.270339,0.411209,-1.186713,...,0.498939,0.422257,1.173478,1.003721,-0.506200,0.872855,-0.669145
116,1.398438,0.307260,-0.438441,1.226302,-0.921602,-2.194652,0.524607,...,1.029320,0.741677,-0.333587,-0.293697,-1.265664,1.545244,0.465039
117,1.871316,-1.697111,-1.380475,-0.437211,0.788573,0.519837,-0.516049,...,1.027785,-0.800984,-0.463563,0.525422,-0.065359,-0.910965,1.808661


If data is given as a flattened array, but the dimensions are known, specify nrow and ncol if a `RangeIndex` is sufficient.

In [38]:
flat_data = np.random.standard_normal((ntime, nrow * ncol))
flat_data

array([[ 0.10736255, -2.23835907, -0.44258974, ...,  1.59837593,
        -0.10850612,  0.32792432],
       [ 1.52681686, -1.32286405, -0.44938048, ...,  1.19486908,
        -0.55627297,  1.31950942],
       [-0.9699366 , -0.38318326, -1.38516173, ...,  0.58044119,
         0.73818326,  0.76395755],
       ...,
       [-1.52831165, -0.75819518,  1.8026957 , ..., -1.06417958,
        -0.9744305 , -0.37624271],
       [ 1.11541103, -0.4739205 , -0.18387137, ...,  1.09843713,
        -0.9727826 , -0.36549961],
       [ 1.62077886, -0.21600267, -0.23855842, ...,  0.25857846,
         0.63997084, -1.16251655]])

In [39]:
cube = ld.DataCube(
    flat_data,
    time_indices={"time": times},
    nrow=nrow,
    ncol=ncol,
)
pd.DataFrame(cube)

,series,0,1,2,3,4,5,6,...,113,114,115,116,117,118,119
,row,0,0,0,0,0,0,0,...,9,9,9,9,9,9,9
,col,0,1,2,3,4,5,6,...,5,6,7,8,9,10,11
time_index,time,,,,,,,,,,,,,,,
0,0.000000,0.107363,-2.238359,-0.442590,0.862550,0.493144,0.899800,-1.251327,...,0.563115,1.477255,-0.831991,1.205002,1.598376,-0.108506,0.327924
1,0.201681,1.526817,-1.322864,-0.449380,-1.335627,0.049743,-0.158684,-0.781415,...,-0.289936,0.400354,-0.599175,-3.011555,1.194869,-0.556273,1.319509
2,0.403361,-0.969937,-0.383183,-1.385162,-1.791713,2.135984,1.678557,1.364678,...,0.590199,1.892906,-1.852756,0.368149,0.580441,0.738183,0.763958
3,0.605042,-0.141658,0.424407,0.939711,-1.268420,1.108402,0.554446,-1.022036,...,-0.452129,-0.149505,-0.459804,1.841598,-0.515234,0.722457,2.187158
4,0.806723,-0.290675,0.723259,-0.179197,-0.775285,-0.851192,0.371850,-1.623399,...,0.607752,-0.337033,0.640671,-0.367987,-1.905231,0.642357,-0.789474
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,23.193277,2.258808,0.947360,-1.334711,1.436611,-0.890944,2.072898,-0.507347,...,-1.900360,0.874428,0.611042,0.748984,0.280635,-0.244457,0.110246


## 2.3 - From a pandas Index object


In [42]:
# Indices as coordinates
row_arr = np.repeat(np.arange(100, 100 + nrow), ncol)
row_index = pd.Index(row_arr, name="row_index")
col_arr = np.tile(np.arange(200, 200 + ncol), nrow)
col_index = pd.Index(col_arr, name="col_index")
series_range = pd.RangeIndex(nrow * ncol, name="series")
columns = pd.MultiIndex.from_arrays([series_range, row_index, col_index])
np.array(list(zip(row_index, col_index)))

array([[100, 200],
       [100, 201],
       [100, 202],
       ...,
       [109, 209],
       [109, 210],
       [109, 211]])

In [43]:
cube = ld.DataCube(
    data,
    time_indices={"time": times},
    columns=columns,
)
pd.DataFrame(cube)

,series,0,1,2,3,4,5,6,...,113,114,115,116,117,118,119
,row_index,100,100,100,100,100,100,100,...,109,109,109,109,109,109,109
,col_index,200,201,202,203,204,205,206,...,205,206,207,208,209,210,211
time_index,time,,,,,,,,,,,,,,,
0,0.000000,0.451025,-0.012730,0.118395,-0.123785,0.368806,0.812296,-1.491025,...,0.542614,0.538944,1.126243,-0.693383,0.168361,-1.926612,-0.747363
1,0.201681,-0.809616,0.694303,1.738103,-0.846800,-0.106610,-1.089616,-0.010108,...,0.468733,0.960319,1.080917,-1.763136,0.581766,-0.629591,-1.067231
2,0.403361,-2.159283,-0.347606,-1.886857,1.021704,0.159404,0.452730,-0.070942,...,0.285698,0.575760,-1.265924,-0.829648,0.758606,0.696689,0.255458
3,0.605042,0.808381,1.468899,-1.657832,1.457362,-0.572793,1.401293,-0.894599,...,1.115110,0.585800,0.804090,1.805249,-2.014900,0.644703,0.118226
4,0.806723,-0.116969,0.485642,-1.661645,-0.095260,0.652312,-1.422831,0.398717,...,2.293649,0.156011,-0.579476,1.150948,-0.761896,-1.584643,0.281398
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,23.193277,2.111950,1.555535,0.472429,0.235726,0.270339,0.411209,-1.186713,...,0.498939,0.422257,1.173478,1.003721,-0.506200,0.872855,-0.669145


## 2.4 - Combining `columns`, `row_indices`, and `col_indices`
Using all available keywords is supported and an easy way to add new column indices to an existing set of columns while keeping track of which pertain to rows and columns. Again, though, the given `columns` need to have the string "row" or "col" in their names to be interpreted correctly as rows and columns for `Cube` objects.

In [ ]:
cube = ld.DataCube(
    data,
    time_indices={"time": times},
    row_indices={"row_pos": row_pos},
    col_indices={"col_pos": col_pos},
    columns=columns,
)
pd.DataFrame(cube)

,series,0,1,2,3,4,5,6,...,113,114,115,116,117,118,119
,row_pos,166.5,166.5,166.5,166.5,166.5,166.5,166.5,...,182.7,182.7,182.7,182.7,182.7,182.7,182.7
,row_index,100,100,100,100,100,100,100,...,109,109,109,109,109,109,109
,col_pos,158.65,159.45,160.25,161.05,161.85,162.65,163.45,...,162.65,163.45,164.25,165.05,165.85,166.65,167.45
,col_index,200,201,202,203,204,205,206,...,205,206,207,208,209,210,211
time_index,time,,,,,,,,,,,,,,,
0,0.000000,0.610027,-1.052723,1.210963,0.645024,-0.701061,-0.453976,-1.276911,...,0.142681,-1.096051,-0.111062,0.359572,-0.074548,-0.166157,-0.227028
1,0.201681,1.748926,0.423949,-0.976110,-0.119464,0.134434,0.852955,0.024060,...,-1.431003,-1.045449,0.432927,-1.173019,-0.626421,-0.416872,-0.088833
2,0.403361,0.754831,1.187173,-0.270114,0.700425,-1.444302,-0.466292,1.057146,...,0.469325,0.288205,-2.455548,0.527181,-0.523647,-0.200747,0.160451
3,0.605042,-0.377059,-0.199929,0.204244,-0.146593,-2.131789,-1.743305,-0.462536,...,1.249152,-0.101836,-0.386387,0.824270,0.765963,-0.634425,0.959132
4,0.806723,2.322809,-0.713305,-0.655203,0.391199,1.086904,-1.203263,0.594491,...,1.126163,0.592046,-0.073098,-0.873564,-0.680229,-0.469045,-1.431175


# 3 - `Cube` from a pandas DataFrame

It may be convenient to create a pandas `DataFrame` at first to wrangle data and set indices and columns. It is possible to create a `Cube` from a pandas `DataFrame` using the `from_pandas` class method.

In [ ]:
df = pd.DataFrame(series_data, index=times)
df.index.name = "time_index"

In [ ]:
# Indices as coordinates
row_arr = np.repeat(np.arange(100, 100 + nrow), ncol)
row_index = pd.Index(row_arr, name="row_index")
col_arr = np.tile(np.arange(200, 200 + ncol), nrow)
col_index = pd.Index(col_arr, name="col_index")
series_range = pd.RangeIndex(nrow * ncol, name="series")
columns = pd.MultiIndex.from_arrays([series_range, row_index, col_index])

df = pd.DataFrame(data.reshape(120, 120), index=times, columns=columns)
df.index.name = "times"
df

series,0,1,2,3,4,5,6,...,113,114,115,116,117,118,119
row_index,100,100,100,100,100,100,100,...,109,109,109,109,109,109,109
col_index,200,201,202,203,204,205,206,...,205,206,207,208,209,210,211
times,,,,,,,,,,,,,,,
0.000000,0.451025,-0.012730,0.118395,-0.123785,0.368806,0.812296,-1.491025,...,0.542614,0.538944,1.126243,-0.693383,0.168361,-1.926612,-0.747363
0.201681,-0.809616,0.694303,1.738103,-0.846800,-0.106610,-1.089616,-0.010108,...,0.468733,0.960319,1.080917,-1.763136,0.581766,-0.629591,-1.067231
0.403361,-2.159283,-0.347606,-1.886857,1.021704,0.159404,0.452730,-0.070942,...,0.285698,0.575760,-1.265924,-0.829648,0.758606,0.696689,0.255458
0.605042,0.808381,1.468899,-1.657832,1.457362,-0.572793,1.401293,-0.894599,...,1.115110,0.585800,0.804090,1.805249,-2.014900,0.644703,0.118226
0.806723,-0.116969,0.485642,-1.661645,-0.095260,0.652312,-1.422831,0.398717,...,2.293649,0.156011,-0.579476,1.150948,-0.761896,-1.584643,0.281398
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23.193277,2.111950,1.555535,0.472429,0.235726,0.270339,0.411209,-1.186713,...,0.498939,0.422257,1.173478,1.003721,-0.506200,0.872855,-0.669145


In [49]:
cube_df = ld.DataCube.from_pandas(
    df,
    nrow=10,  # DataFrames are 2D, nrow and ncol must be specified
    ncol=12,
    # uncertainty=data_err,  # DataFrames have no standard way to store uncertainty, if desired it must be given
    # **kwargs  # any additional metadata to provide
)
cube_df

📘 DataCube (120, 10, 12), Uncertainty: False

In [52]:
cube_df.describe_cube()

📘 DataCube (120, 10, 12), Uncertainty: False (ntime, nrow, ncol)
pd.DataFrame shape: (120, 120)


Time indices available: ['time_index', 'times']
  time_index  :	[  0   1 ... 118 119]
  times       :	[ 0.          0.20168067 ... 23.79831933 24.        ]

Number of unique 'series': 120

Row names: ['row_index']
  row_index   :	[100 100 ... 109 109]

Column names: ['col_index']
  col_index   :	[200 201 ... 210 211]
